<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/main/phase_01_data_exploration/data_exploration_of_sub_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading and Extracting COCO 2017 Dataset
This notebook downloads the annotations, training dataset, and validation dataset from COCO 2017, and then extracts them.

# Download The data set + Unzip it.

In [ ]:
# 1. Download annotations, train images, and validation images
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/train2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip

annotations_trainva 100%[===================>] 241.19M  20.9MB/s    in 13s     
train2017.zip       100%[===================>]  18.01G  20.5MB/s    in 15m 23s 
val2017.zip         100%[===================>] 777.80M  21.3MB/s    in 38s     


In [ ]:
# 2. Extract all downloaded zip files
print("Extracting annotations...")
!unzip -q annotations_trainval2017.zip

Extracting annotations...


In [ ]:
print("Extracting training dataset (this may take a while)...")
!unzip -q train2017.zip

Extracting training dataset (this may take a while)...


In [ ]:
print("Extracting validation dataset...")
!unzip -q val2017.zip

Extracting validation dataset...


# Data Exploration
Calculation of metrics

## Distribution of object classes

In [9]:
from collections import Counter
import json
import os

# Google Colab path definitions
INSTANCES_TRAIN_FILE = os.path.join("annotations", "instances_train2017.json")
INSTANCES_VAL_FILE = os.path.join("annotations", "instances_val2017.json")

TEST_SPLIT_FILE = os.path.join("splits", "test_100.json")
VAL_SPLIT_FILE = os.path.join("splits", "val_100.json")
KB_SPLIT_FILE = os.path.join("splits", "kb_10k.json")


def read_json(file_path):
    """Utility function to load a JSON file."""
    with open(file_path, "r") as f:
        return json.load(f)


def build_image_to_categories_map(instances_data):
    """Maps image_id -> list of object category IDs present in that image."""
    image_to_categories = {}
    for ann in instances_data["annotations"]:
        image_id = ann["image_id"]
        cat_id = ann["category_id"]
        image_to_categories.setdefault(image_id, []).append(cat_id)
    return image_to_categories


def count_top_classes_for_split(
    split_data, image_to_categories_map, category_name_map, top_k=50
):
    """Calculates object instance frequencies for a split and returns Top K."""
    class_counts = Counter()
    for item in split_data:
        image_id = item["image_id"]
        # Extract the catogories existing in the current image
        cat_ids = image_to_categories_map.get(image_id, [])
        for cid in cat_ids:
            # Get category name
            cat_name = category_name_map.get(cid, f"Unknown_{cid}")
            # Increase category count
            class_counts[cat_name] += 1
    # order (descend) and return top k
    return class_counts.most_common(top_k)


def display_top_classes(split_name, top_classes):
    """Prints a formatted table of Top 50 classes and their occurrence counts."""
    print(f"\n==================================================")
    print(f"   Top 50 Object Classes: {split_name}")
    print(f"==================================================")
    print(f"{'Rank':<6} | {'Category Name':<22} | {'Occurrences':<12}")
    print("-" * 48)
    for rank, (cat_name, count) in enumerate(top_classes, start=1):
        print(f"{rank:<6} | {cat_name:<22} | {count:<12}")


def run_object_distribution_analysis():
    print("Loading COCO instance annotations...")
    train_instances = read_json(INSTANCES_TRAIN_FILE)
    val_instances = read_json(INSTANCES_VAL_FILE)

    # Build category ID to Name map
    category_name_map = {
        cat["id"]: cat["name"] for cat in train_instances["categories"]
    }

    # Build image_id -> category_ids lookup maps
    train_image_to_cats = build_image_to_categories_map(train_instances)
    val_image_to_cats = build_image_to_categories_map(val_instances)

    # 1. Process Test Split (sourced from val2017)
    if os.path.exists(TEST_SPLIT_FILE):
        test_data = read_json(TEST_SPLIT_FILE)
        top_test = count_top_classes_for_split(
            test_data, val_image_to_cats, category_name_map
        )
        display_top_classes("Test Subset (test_100.json)", top_test)
    else:
        print(f"⚠️ File not found: {TEST_SPLIT_FILE}")

    # 2. Process Validation Split (sourced from train2017)
    if os.path.exists(VAL_SPLIT_FILE):
        val_data = read_json(VAL_SPLIT_FILE)
        top_val = count_top_classes_for_split(
            val_data, train_image_to_cats, category_name_map
        )
        display_top_classes("Validation Subset (val_100.json)", top_val)
    else:
        print(f"⚠️ File not found: {VAL_SPLIT_FILE}")

    # 3. Process Knowledge Base Split (sourced from train2017)
    if os.path.exists(KB_SPLIT_FILE):
        kb_data = read_json(KB_SPLIT_FILE)
        top_kb = count_top_classes_for_split(
            kb_data, train_image_to_cats, category_name_map
        )
        display_top_classes("Knowledge Base Subset (kb_10k.json)", top_kb)
    else:
        print(f"⚠️ File not found: {KB_SPLIT_FILE}")


if __name__ == "__main__":
    run_object_distribution_analysis()

Loading COCO instance annotations...

   Top 50 Object Classes: Test Subset (test_100.json)
Rank   | Category Name          | Occurrences 
------------------------------------------------
1      | person                 | 85          
2      | vase                   | 41          
3      | bottle                 | 30          
4      | toilet                 | 29          
5      | book                   | 28          
6      | clock                  | 17          
7      | zebra                  | 17          
8      | bus                    | 16          
9      | refrigerator           | 11          
10     | teddy bear             | 10          
11     | cat                    | 9           
12     | giraffe                | 8           
13     | potted plant           | 8           
14     | remote                 | 8           
15     | sheep                  | 7           
16     | truck                  | 7           
17     | couch                  | 7           
18     | car 

## Scene complexity

In [10]:
import json
import os
import statistics

# Paths for Google Colab environment
INSTANCES_TRAIN_FILE = os.path.join("annotations", "instances_train2017.json")
INSTANCES_VAL_FILE = os.path.join("annotations", "instances_val2017.json")

TEST_SPLIT_FILE = os.path.join("splits", "test_100.json")
VAL_SPLIT_FILE = os.path.join("splits", "val_100.json")
KB_SPLIT_FILE = os.path.join("splits", "kb_10k.json")


def read_json(file_path):
    """Utility function to load a JSON file."""
    with open(file_path, "r") as f:
        return json.load(f)


def build_image_object_counts(instances_data):
    """Maps image_id -> total count of annotated object instances."""
    object_counts = {}
    for ann in instances_data["annotations"]:
        image_id = ann["image_id"]
        object_counts[image_id] = object_counts.get(image_id, 0) + 1
    return object_counts


def analyze_split_complexity(split_name, split_data, image_object_counts):
    """Calculates statistical distribution and simple vs. complex breakdown."""
    counts = []
    for item in split_data:
        image_id = item["image_id"]
        # Retrieve total instances for this image (default to 0 if none found)
        count = image_object_counts.get(image_id, 0)
        counts.append(count)

    total_images = len(counts)
    if total_images == 0:
        print(f"⚠️ No images found in {split_name}")
        return

    # Categorical breakdown
    simple_count = sum(1 for c in counts if 1 <= c <= 2)
    complex_count = sum(1 for c in counts if c >= 3)
    other_count = sum(1 for c in counts if c == 0)

    simple_pct = (simple_count / total_images) * 100
    complex_pct = (complex_count / total_images) * 100

    # Statistical distribution
    mean_val = statistics.mean(counts)
    median_val = statistics.median(counts)
    min_val = min(counts)
    max_val = max(counts)

    # Display results
    print(f"\n==================================================")
    print(f"   Scene Complexity Analysis: {split_name}")
    print(f"==================================================")
    print(f"Total Images Analyzed: {total_images}")
    print("-" * 50)
    print("📊 Categorical Breakdown:")
    print(f"  • Simple Scenes (1-2 objects): {simple_count:<5} ({simple_pct:.2f}%)")
    print(f"  • Complex Scenes (>=3 objects): {complex_count:<5} ({complex_pct:.2f}%)")
    if other_count > 0:
        print(f"  • Unannotated Scenes (0 objects): {other_count:<5}")
    print("-" * 50)
    print("📈 Statistical Metrics:")
    print(f"  • Mean (Average) Objects / Image: {mean_val:.2f}")
    print(f"  • Median Objects / Image:        {median_val:.2f}")
    print(f"  • Min Objects in an Image:       {min_val}")
    print(f"  • Max Objects in an Image:       {max_val}")


def run_scene_complexity_analysis():
    print("Loading COCO instance annotations...")
    train_instances = read_json(INSTANCES_TRAIN_FILE)
    val_instances = read_json(INSTANCES_VAL_FILE)

    # Build image_id -> object count maps
    train_counts_map = build_image_object_counts(train_instances)
    val_counts_map = build_image_object_counts(val_instances)

    # 1. Process Test Split (sourced from val2017)
    if os.path.exists(TEST_SPLIT_FILE):
        test_data = read_json(TEST_SPLIT_FILE)
        analyze_split_complexity("Test Subset (test_100.json)", test_data, val_counts_map)
    else:
        print(f"⚠️ File not found: {TEST_SPLIT_FILE}")

    # 2. Process Validation Split (sourced from train2017)
    if os.path.exists(VAL_SPLIT_FILE):
        val_data = read_json(VAL_SPLIT_FILE)
        analyze_split_complexity("Validation Subset (val_100.json)", val_data, train_counts_map)
    else:
        print(f"⚠️ File not found: {VAL_SPLIT_FILE}")

    # 3. Process Knowledge Base Split (sourced from train2017)
    if os.path.exists(KB_SPLIT_FILE):
        kb_data = read_json(KB_SPLIT_FILE)
        analyze_split_complexity("Knowledge Base Subset (kb_10k.json)", kb_data, train_counts_map)
    else:
        print(f"⚠️ File not found: {KB_SPLIT_FILE}")


if __name__ == "__main__":
    run_scene_complexity_analysis()

Loading COCO instance annotations...

   Scene Complexity Analysis: Test Subset (test_100.json)
Total Images Analyzed: 100
--------------------------------------------------
📊 Categorical Breakdown:
  • Simple Scenes (1-2 objects): 52    (52.00%)
  • Complex Scenes (>=3 objects): 48    (48.00%)
--------------------------------------------------
📈 Statistical Metrics:
  • Mean (Average) Objects / Image: 4.63
  • Median Objects / Image:        2.00
  • Min Objects in an Image:       1
  • Max Objects in an Image:       28

   Scene Complexity Analysis: Validation Subset (val_100.json)
Total Images Analyzed: 100
--------------------------------------------------
📊 Categorical Breakdown:
  • Simple Scenes (1-2 objects): 30    (30.00%)
  • Complex Scenes (>=3 objects): 68    (68.00%)
  • Unannotated Scenes (0 objects): 2    
--------------------------------------------------
📈 Statistical Metrics:
  • Mean (Average) Objects / Image: 7.48
  • Median Objects / Image:        4.00
  • Min Objec

## Textual statistics

In [11]:
from collections import Counter
import json
import os
import statistics
import nltk

# Ensure NLTK resources are downloaded
# We are going to use NLTK to tokenize the captions
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

# Paths for Google Colab environment
TEST_SPLIT_FILE = os.path.join("splits", "test_100.json")
VAL_SPLIT_FILE = os.path.join("splits", "val_100.json")
KB_SPLIT_FILE = os.path.join("splits", "kb_10k.json")


def read_json(file_path):
    """Utility function to load a JSON file."""
    with open(file_path, "r") as f:
        return json.load(f)


def map_pos_tag(tag):
    """Map Penn Treebank POS tags to simplified syntactic categories."""
    if tag.startswith("NN"):
        return "Nouns (NN)"
    elif tag.startswith("VB"):
        return "Verbs (VB)"
    elif tag.startswith("JJ"):
        return "Adjectives (JJ)"
    elif tag == "IN":
        return "Prepositions / Subordinating Conjunctions (IN)"
    elif tag.startswith("RB"):
        return "Adverbs (RB)"
    elif tag in ["DT", "PDT", "WDT"]:
        return "Determiners (DT)"
    elif tag in ["PRP", "PRP$", "WP", "WP$"]:
        return "Pronouns (PRP)"
    else:
        return "Others"


def analyze_textual_statistics(split_name, split_data):
    """Calculates caption length distribution, vocabulary size, and POS distribution."""
    all_captions = []

    # Extract all reference captions
    for item in split_data:
        all_captions.extend(item.get("captions", []))

    total_captions = len(all_captions)
    if total_captions == 0:
        print(f"⚠️ No captions found in {split_name}")
        return

    caption_lengths = []
    all_tokens = []
    pos_counts = Counter()

    for caption in all_captions:
        # Tokenization (lowercased)
        tokens = nltk.word_tokenize(caption.lower())
        # Filter tokens and keep only real words, execlude ponctuations
        word_tokens = [t for t in tokens if t.isalnum()]

        caption_lengths.append(len(word_tokens))
        all_tokens.extend(word_tokens)

        # POS Tagging
        # Tag each word
        tagged_words = nltk.pos_tag(word_tokens)
        for _, tag in tagged_words:
            # counts occurence per tag
            simplified_pos = map_pos_tag(tag)
            pos_counts[simplified_pos] += 1

    total_words = len(all_tokens)
    # remove duplications using set to calculate vocab size
    vocab_size = len(set(all_tokens))

    # Length metrics
    mean_len = statistics.mean(caption_lengths)
    median_len = statistics.median(caption_lengths)
    min_len = min(caption_lengths)
    max_len = max(caption_lengths)

    # Display results
    print(f"\n==================================================")
    print(f"   Textual Statistics Analysis: {split_name}")
    print(f"==================================================")
    print(f"Total Captions Analyzed: {total_captions}")
    print(f"Total Word Tokens:       {total_words}")
    print(f"Unique Vocabulary Size:  {vocab_size} unique words")
    print("-" * 50)
    print("📏 Caption Length Distribution (Words / Caption):")
    print(f"  • Mean (Average) Length: {mean_len:.2f} words")
    print(f"  • Median Length:         {median_len:.2f} words")
    print(f"  • Min Length:            {min_len} words")
    print(f"  • Max Length:            {max_len} words")
    print("-" * 50)
    print("🏷️ Part-of-Speech (POS) Distribution:")
    print(f"{'POS Category':<42} | {'Count':<10} | {'Percentage':<10}")
    print("-" * 68)
    for pos_cat, count in pos_counts.most_common():
        pct = (count / total_words) * 100 if total_words > 0 else 0
        print(f"{pos_cat:<42} | {count:<10} | {pct:.2f}%")


def run_textual_statistics_analysis():
    print("Initializing NLTK resources and loading dataset splits...")

    # 1. Process Test Split
    if os.path.exists(TEST_SPLIT_FILE):
        test_data = read_json(TEST_SPLIT_FILE)
        analyze_textual_statistics("Test Subset (test_100.json)", test_data)
    else:
        print(f"⚠️ File not found: {TEST_SPLIT_FILE}")

    # 2. Process Validation Split
    if os.path.exists(VAL_SPLIT_FILE):
        val_data = read_json(VAL_SPLIT_FILE)
        analyze_textual_statistics("Validation Subset (val_100.json)", val_data)
    else:
        print(f"⚠️ File not found: {VAL_SPLIT_FILE}")

    # 3. Process Knowledge Base Split
    if os.path.exists(KB_SPLIT_FILE):
        kb_data = read_json(KB_SPLIT_FILE)
        analyze_textual_statistics("Knowledge Base Subset (kb_10k.json)", kb_data)
    else:
        print(f"⚠️ File not found: {KB_SPLIT_FILE}")


if __name__ == "__main__":
    run_textual_statistics_analysis()

Initializing NLTK resources and loading dataset splits...

   Textual Statistics Analysis: Test Subset (test_100.json)
Total Captions Analyzed: 500
Total Word Tokens:       5140
Unique Vocabulary Size:  934 unique words
--------------------------------------------------
📏 Caption Length Distribution (Words / Caption):
  • Mean (Average) Length: 10.28 words
  • Median Length:         10.00 words
  • Min Length:            7 words
  • Max Length:            30 words
--------------------------------------------------
🏷️ Part-of-Speech (POS) Distribution:
POS Category                               | Count      | Percentage
--------------------------------------------------------------------
Nouns (NN)                                 | 1774       | 34.51%
Determiners (DT)                           | 1061       | 20.64%
Prepositions / Subordinating Conjunctions (IN) | 815        | 15.86%
Verbs (VB)                                 | 585        | 11.38%
Adjectives (JJ)                         